# Imports

In [18]:
import os, time, psutil, pickle, sys
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import pynvml
from sklearn.metrics.pairwise import cosine_similarity

import open_clip
from transformers import CLIPProcessor, CLIPModel  


# Config

In [19]:
BASE_DIR = os.path.join(os.getcwd(), "TFE_Data")
DATASET = "Flickr8k"
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")
RESULTS_DIR = os.path.join(BASE_DIR, "Results_Multimodal")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Running on:", DEVICE)

Running on: cuda


# Load Dataset

In [20]:
# Load dataset metadata
df_path = os.path.join(DATASETS_DIR, f"df_{DATASET}.pkl")
df = pd.read_pickle(df_path)

IMAGE_PATHS = df["image_path"].tolist()      # 600 images
CAPTIONS_LIST = df["captions"].tolist()    # list of 5 captions per image


In [21]:
# Flatten → 3000 captions
CAPTIONS = [cap for caps in CAPTIONS_LIST for cap in caps]
image_id_for_caption = np.repeat(np.arange(len(IMAGE_PATHS)), 5)

print("Images:", len(IMAGE_PATHS))
print("Captions:", len(CAPTIONS))

Images: 8091
Captions: 40455


# Load Models

In [ ]:
def load_multimodal_model(model_name):

    # ------------------------------------------------------------
    # 1. HuggingFace CLIP (baseline)
    # ------------------------------------------------------------
    if model_name == "clip":
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
        return model_name, (model, processor), None

    # ------------------------------------------------------------
    # 2. OpenCLIP ViT-L/14 (strong baseline)
    # ------------------------------------------------------------
    elif model_name == "openclip_l14":
        model, _, preprocess = open_clip.create_model_and_transforms(
            "ViT-L-14", pretrained="openai"
        )
        tokenizer = open_clip.get_tokenizer("ViT-L-14")
        model = model.to(DEVICE)
        return model_name, (model, preprocess, tokenizer), None


# GreenAI Metrics

In [23]:
def measure_memory():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def measure_gpu_memory():
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    return pynvml.nvmlDeviceGetMemoryInfo(handle).used / 1024**2

def measure_gpu_utilization():
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    return pynvml.nvmlDeviceGetUtilizationRates(handle).gpu

def estimate_energy(flops, gflops_per_s=35000, power_w=300):
    if flops is None:
        return None
    exec_time_s = flops / (gflops_per_s * 1e9)
    return exec_time_s * power_w

# XAI Methods

In [24]:
def save_vision_saliency(activations, img_path, model_name):
    act = activations.detach().cpu()

    # ViT tokens → saliency
    tokens = act[:, 1:, :]
    weights = torch.norm(tokens, dim=-1, keepdim=True)
    weighted = torch.sum(tokens * weights, dim=-1)

    side = int(np.sqrt(tokens.shape[1]))
    heatmap = weighted.view(side, side).numpy()

    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap) if np.max(heatmap) > 0 else 1

    out_dir = os.path.join(RESULTS_DIR, DATASET, model_name, "Saliency_Vision")
    os.makedirs(out_dir, exist_ok=True)

    name = os.path.splitext(os.path.basename(img_path))[0]
    np.save(os.path.join(out_dir, f"{name}.npy"), heatmap)

def save_text_saliency(tokens, scores, idx, model_name):
    out_dir = os.path.join(RESULTS_DIR, DATASET, model_name, "Saliency_Text")
    os.makedirs(out_dir, exist_ok=True)

    np.save(os.path.join(out_dir, f"text_{idx}.npy"), {
        "tokens": tokens,
        "scores": scores
    })

# Extract Embeddings

## Multimodal

In [ ]:
def extract_embeddings(model_name, model_bundle):

    # ------------------------------------------------------------
    # 1. CLIP (HuggingFace)
    # ------------------------------------------------------------
    if model_name == "clip":
        model, processor = model_bundle
        vision_model = model.vision_model
        text_model = model.text_model

        # Vision hook for XAI
        last_acts = None
        def hook_fn(module, inp, out):
            nonlocal last_acts
            last_acts = out

        hook = vision_model.encoder.layers[-1].register_forward_hook(hook_fn)

        # Vision embeddings
        vision_feats = []
        for img_path in tqdm(IMAGE_PATHS, desc="CLIP Vision"):
            img = Image.open(img_path).convert("RGB")
            inputs = processor(images=img, return_tensors="pt").to(DEVICE)

            with torch.no_grad():
                out = vision_model(pixel_values=inputs["pixel_values"])
                emb = model.visual_projection(out.pooler_output)

            vision_feats.append(emb.cpu().numpy())

            if last_acts is not None:
                save_vision_saliency(last_acts, img_path, "clip")

        hook.remove()
        vision_feats = np.vstack(vision_feats)

        # Text embeddings
        text_feats = []
        idx = 0
        for cap in tqdm(CAPTIONS, desc="CLIP Text"):
            inputs = processor(text=cap, return_tensors="pt", padding=True).to(DEVICE)

            with torch.no_grad():
                out = text_model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"]
                )
                emb = model.text_projection(out.pooler_output)

            text_feats.append(emb.cpu().numpy())

            tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
            scores = emb.norm(dim=-1).cpu().numpy()
            save_text_saliency(tokens, scores, idx, "clip")
            idx += 1

        text_feats = np.vstack(text_feats)
        return vision_feats, text_feats


    # ------------------------------------------------------------
    # 2–5. OpenCLIP models
    # ------------------------------------------------------------
    elif model_name in ["openclip_l14"]:
        model, preprocess, tokenizer = model_bundle

        # Vision embeddings
        vision_feats = []
        for img_path in tqdm(IMAGE_PATHS, desc=f"{model_name.upper()} Vision"):
            img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                emb = model.encode_image(img)

            vision_feats.append(emb.cpu().numpy())

        vision_feats = np.vstack(vision_feats)

        # Text embeddings
        text_feats = []
        for cap in tqdm(CAPTIONS, desc=f"{model_name.upper()} Text"):
            tokens = tokenizer(cap).to(DEVICE)

            with torch.no_grad():
                emb = model.encode_text(tokens)

            text_feats.append(emb.cpu().numpy())

        text_feats = np.vstack(text_feats)
        return vision_feats, text_feats

    else:
        raise ValueError(f"Unknown model: {model_name}")


# Execute

In [26]:
def run_multimodal(model_name):
    print(f"\n=== Running {model_name.upper()} ===")

    start = time.time()
    mem_before = measure_memory()
    gpu_before = measure_gpu_memory()

    model_name, model_bundle, processor = load_multimodal_model(model_name)
    vision_feats, text_feats = extract_embeddings(model_name, model_bundle)

    exec_time = time.time() - start
    mem_used = measure_memory() - mem_before
    gpu_used = measure_gpu_memory() - gpu_before

    # Normalize
    vision_feats /= np.linalg.norm(vision_feats, axis=1, keepdims=True)
    text_feats /= np.linalg.norm(text_feats, axis=1, keepdims=True)

    # Save
    out_dir = os.path.join(RESULTS_DIR, DATASET, model_name)
    os.makedirs(out_dir, exist_ok=True)

    np.save(os.path.join(out_dir, "vision_embeddings.npy"), vision_feats)
    np.save(os.path.join(out_dir, "text_embeddings.npy"), text_feats)

    return {
        "Model": model_name,
        "Time_s": exec_time,
        "Memory_MB": mem_used,
        "GPU_Memory_MB": gpu_used,
        "GPU_Util_percent": measure_gpu_utilization(),
        "CPU_Util_percent": psutil.cpu_percent(),
        "Vision_Dim": vision_feats.shape[1],
        "Text_Dim": text_feats.shape[1],
    }


In [27]:
def evaluate(model_name):
    out_dir = os.path.join(RESULTS_DIR, DATASET, model_name)
    vision = np.load(os.path.join(out_dir, "vision_embeddings.npy"))
    text = np.load(os.path.join(out_dir, "text_embeddings.npy"))

    def recall_i2t(k):
        sims = cosine_similarity(vision, text)
        correct = 0
        for i in range(len(vision)):
            top_k = np.argsort(-sims[i])[:k]
            if any(image_id_for_caption[j] == i for j in top_k):
                correct += 1
        return correct / len(vision)

    def recall_t2i(k):
        sims = cosine_similarity(text, vision)
        correct = 0
        for i in range(len(text)):
            top_k = np.argsort(-sims[i])[:k]
            if image_id_for_caption[i] in top_k:
                correct += 1
        return correct / len(text)

    print(f"\n=== Retrieval for {model_name.upper()} ===")
    print("I2T R@1:", recall_i2t(1))
    print("I2T R@5:", recall_i2t(5))
    print("T2I R@1:", recall_t2i(1))
    print("T2I R@5:", recall_t2i(5))


In [ ]:
models = ["openclip_l14", "clip"]

for m in models:
    metrics = run_multimodal(m)
    print(metrics)
    evaluate(m)




=== Running OPENCLIP_L14 ===


/home/aysel/tfe/.venv/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
OPENCLIP_L14 Text: 100%|██████████| 40455/40455 [02:26<00:00, 275.51it/s]


{'Model': 'openclip_l14', 'Time_s': 289.89488101005554, 'Memory_MB': 209.93359375, 'GPU_Memory_MB': 0.0, 'GPU_Util_percent': 45, 'CPU_Util_percent': 10.7, 'Vision_Dim': 768, 'Text_Dim': 768}

=== Retrieval for OPENCLIP_L14 ===
I2T R@1: 0.5438141144481523
I2T R@5: 0.7704857248794957
T2I R@1: 0.3612161661104931
T2I R@5: 0.6026943517488568

=== Running EVA ===
Available EVA02 pretrained tags: ['merged2b_s4b_b131k']


open_clip_model.safetensors:   0%|          | 0.00/856M [00:00<?, ?B/s]

EVA Text: 100%|██████████| 40455/40455 [02:26<00:00, 275.35it/s]


...

COCA Text: 100%|██████████| 40455/40455 [02:32<00:00, 265.51it/s]


{'Model': 'coca', 'Time_s': 325.21052169799805, 'Memory_MB': 1155.7578125, 'GPU_Memory_MB': 796.0, 'GPU_Util_percent': 7, 'CPU_Util_percent': 26.3, 'Vision_Dim': 768, 'Text_Dim': 768}

=== Retrieval for COCA ===
I2T R@1: 0.6349029786182178
I2T R@5: 0.8452601656161167
T2I R@1: 0.521616611049314
T2I R@5: 0.7579532814238042

=== Running CLIP ===


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP Text: 100%|██████████| 40455/40455 [03:26<00:00, 196.24it/s]


{'Model': 'clip', 'Time_s': 271.32408714294434, 'Memory_MB': 24.3984375, 'GPU_Memory_MB': 44.0, 'GPU_Util_percent': 28, 'CPU_Util_percent': 17.7, 'Vision_Dim': 512, 'Text_Dim': 512}

=== Retrieval for CLIP ===
I2T R@1: 0.4742306266221728
I2T R@5: 0.7048572487949574
T2I R@1: 0.29823260412804353
T2I R@5: 0.5330614262761093
